In [ ]:
import os
import pandas as pd
import numpy as np
import commons as c

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from scipy.stats import wilcoxon
from scipy.stats import mannwhitneyu
from math import sqrt

In [ ]:
csv_path = 'results/dataframes/results_all_mutants.csv'
df = pd.read_csv(csv_path, dtype=c.type_dict)

In [ ]:
df_total = df.melt(
    id_vars=['Algorithm', 'Qubits_number', 'hardware', 'nature', 'metric', 'metric_full', 'hardware_named'],
    value_vars=['ideal_distance', 'noisy_distance'],
    value_name='distance'
)
df_total.loc[df_total['variable'] == 'ideal_distance', 'hardware'] = 'ideal'
df_total.loc[df_total['variable'] == 'ideal_distance', 'hardware_named'] = 'Noiseless'
df_total = df_total.drop(columns=['variable'])

In [ ]:
df["nature"] = df["nature"].map({
    "Equivalent mutant": False,
    "Non-Equivalent mutant": True
})

# RQ1.1: 

In [ ]:
def print_box_plot(df, file_name):

    # Create the scatter plot with the adjusted x-axis values
    fig = px.box(df, 
            y='distance', 
            x="metric_full", 
            color='hardware_named',  
            # title="Boxplot of Distance by noise model and program type",
            labels={"metric_full": "Metric", 'hardware_named': "Simulator", "distance": "Distance"},
            points=False,
            boxmode="group",
            color_discrete_map=c.color_map,
            category_orders={'metric_full': list(c.metric_names.values())}
    )

    # Save the figure
    output_folder = 'results/RQ1/RQ1_1/'
    os.makedirs(output_folder, exist_ok=True)
    c.setup_layout_and_save(fig, output_folder, file_name, yaxis_range=[0, 1])

In [ ]:
df_equiv = df_total[df_total['nature'] == 'Equivalent mutant']
file_name = f'visu_equiv'
print_box_plot(df_equiv, file_name)
  
df_normal = df_total[df_total['nature'] == 'Non-Equivalent mutant']  
file_name = f'visu_normal'
print_box_plot(df_normal, file_name)

In [ ]:
def run_wilcoxon_tests_rq11(df):
    """
    Runs paired Wilcoxon Signed-Rank tests comparing
    noiseless vs noisy distances for each:
    - mutant type
    - metric
    - noise model

    Pairing is implicit: (ideal_distance, noisy_distance) per row.
    """

    results = []

    for nature in df['nature'].unique():
        df_nature = df[df['nature'] == nature]

        for metric in df_nature['metric'].unique():
            df_metric = df_nature[df_nature['metric'] == metric]

            for noise in df_metric['hardware'].unique():
                df_noise = df_metric[df_metric['hardware'] == noise]

                # Drop invalid or identical pairs (Wilcoxon requires differences)
                #paired = df_noise[(df_noise['ideal_distance'] != df_noise['noisy_distance'])]

                res = wilcoxon(df_noise['noisy_distance'], df_noise['ideal_distance'], alternative='two-sided')

                if nature:
                    equiv = "Non-Equivalent mutant"
                else:
                    equiv =  "Equivalent mutant"

                n = len(df_noise)
                W = res.statistic
                r = 1 - (2*W) / (n*(n+1)/2)

                results.append({
                    'nature': equiv,
                    'metric': metric,
                    'noise_model': noise,
                    'wilcoxon_stat': res.statistic,
                    'p_value': res.pvalue,
                    'r': r,
                    #'median_ideal': df_noise['ideal_distance'].median(),
                    #'median_noisy': df_noise['noisy_distance'].median(),
                    'median_diff': (
                        df_noise['noisy_distance'] - df_noise['ideal_distance']
                    ).median()
                })

    return pd.DataFrame(results)


In [ ]:
wilcoxon_results = run_wilcoxon_tests_rq11(df)

In [ ]:
wilcoxon_results = wilcoxon_results.sort_values('r')

print(wilcoxon_results)

# RQ1.2: 

In [ ]:
def add_threshold_line(fig, hw, t, metric, xmin, xmax, color, dash):
    
    threshold_value = c.get_tolerance_values(t, hw)[metric]
    fig.add_shape(
        type="line",
        x0= xmin-0.5,
        x1= xmax+0.5,
        y0=threshold_value,
        y1=threshold_value,
        line=dict(color=color, dash=dash, width=4),
        xref="x",
        yref="y",
    )

    #if t == 'N':
        #print("test N")
        # fig.add_trace(go.Scatter(
        #    x=[None], y=[None],  # Invisible points
        #   mode="lines",
        #    line=dict(color=color, dash=dash),
        #    name=f"Threshold T<sup>{hw}</sup>" #N_{hw}"
        #))
    #else:
    if t != 'N':
        if t == 'I':
            tname = "Noiseless"
        elif t == 'M':
            tname = "Middle"
        elif t == 'A':
            tname = "Above"
        fig.add_trace(go.Scatter(
            x=[None], y=[None],  # Invisible points
            mode="lines",
            line=dict(color=color, dash=dash),
            name=f"Threshold T<sup>{tname}</sup>"
        ))


In [ ]:
def print_box_plot(df, file_name, metric, with_threshold=False):

    colors = px.colors.qualitative.Prism

    # Create the scatter plot with the adjusted x-axis values
    fig = px.box(df,
            y='distance',
            x="hardware_named",
            color='nature',
        #    # title="Boxplot of distance by noise model and mutation type",
            labels={"hardware_named": "Simulator", 'distance': "Distance", 'nature': "Mutant type"},
            points=False,
            boxmode="group",
            color_discrete_sequence=[colors[6], colors[7]])

    # Add the threshold line
    if with_threshold:
        add_threshold_line(fig, '', 'I', metric, 0, 3, colors[4], None)
        add_threshold_line(fig, '', 'M', metric, 1, 3, colors[3], None) #'dash')
        
        for idx, hw in enumerate(c.hardware):
            add_threshold_line(fig, hw, 'N', metric, idx+1, idx+1, colors[2], None)

        fig.add_trace(go.Scatter(
            x=[None], y=[None],  # Invisible points
            mode="lines",
            line=dict(color=colors[2], dash=None),
            name=f"Threshold T<sup>Noisy</sup>" #N_{hw}"
        ))
    
        add_threshold_line(fig, '', 'A', metric, 1, 3, colors[1], None)
        output_folder = 'results/RQ1/RQ1_2/with_thresholds/'
        os.makedirs(output_folder, exist_ok=True)
        c.setup_layout_and_save(fig, output_folder, file_name, yaxis_range=[0, 1]) #, height=700)
    
    else:   
        output_folder = 'results/RQ1/RQ1_2/no_thresholds/'
        os.makedirs(output_folder, exist_ok=True)
        c.setup_layout_and_save(fig, output_folder, file_name, yaxis_range=[0, 1]) #, height=700)

In [ ]:
for m in c.metrics:
    df_metric = df_total[df_total['metric'] == m]
    fig_name = f"Comparison between equivalent and non-equivalent mutant detectability for {m}"
    file_name = f'visu_{m}'
    print_box_plot(df_metric, file_name, c.metrics[m])
    print_box_plot(df_metric, file_name, c.metrics[m], True)

In [ ]:
# More robust version
# U must correspond to the first sample (nonequiv in your case).
# mannwhitneyu(nonequiv, equiv, ...) → correct.
def cliffs_delta_from_u(U, nx, ny):
    """
    Compute Cliff's delta from Mann–Whitney U statistic.

    Parameters
    ----------
    U : float
        Mann–Whitney U statistic for sample x
    nx, ny : int
        Sample sizes

    Returns
    -------
    delta : float
        Cliff's delta
    """
    return (2 * U) / (nx * ny) - 1


In [ ]:
def run_mannwhitney_rq12(df_total):
    results = []

    for metric in df_total['metric'].unique():
        df_metric = df_total[df_total['metric'] == metric]

        for hw in df_metric['hardware'].unique():

            print(f'Mann-Withney {metric} -- {hw}')

            df_hw = df_metric[df_metric['hardware'] == hw]

            equiv = df_hw[df_hw['nature'] == 'Equivalent mutant']['distance']
            nonequiv = df_hw[df_hw['nature'] == 'Non-Equivalent mutant']['distance']

            # Drop NaNs
            equiv = equiv.dropna()
            nonequiv = nonequiv.dropna()

            stat, p = mannwhitneyu(nonequiv, equiv, alternative='two-sided')

            n1, n2 = len(nonequiv), len(equiv)
            n_total = n1 + n2

            # Calculate mean and std of U under H0
            u_mean = n1 * n2 / 2
            u_std = sqrt(n1 * n2 * (n_total + 1) / 12)

            # Z-score
            z = (stat - u_mean) / u_std

            # Effect size r
            r = abs(z) / sqrt(n_total) # Source: R/wilcox_effsize.R
            abs_r = abs(r) if pd.notna(r) else np.nan

            #print(f'Cliff {metric} -- {hw}')
            #delta = cliffs_delta_from_u(stat, len(nonequiv), len(equiv))

            results.append({
                'metric': metric,
                'hardware': hw,
                'n_equiv': len(equiv),
                'n_nonequiv': len(nonequiv),
                'mw_stat': stat,
                'p_value': p,
                'effect size': abs_r,
                #'cliffs_delta': delta,
                'median_equiv': equiv.median(),
                'median_nonequiv': nonequiv.median()
            })

    return pd.DataFrame(results)


In [ ]:
mw_results = run_mannwhitney_rq12(df_total)

mw_results = mw_results.sort_values(
    ['metric', 'hardware', 'p_value']
)

print(mw_results)


# RQ1.3: Confusion matrices


In [ ]:
def confusion_matrix(df, col1, col2):
    # Create a confusion matrix DataFrame
    conf_matrix = pd.DataFrame(index=['True', 'False'], columns=['True', 'False'])
    
    # Calculate the count of each pair
    true_true = ((df[col1] == True) & (df[col2] == True)).sum()
    false_false = ((df[col1] == False) & (df[col2] == False)).sum()
    true_false = ((df[col1] == True) & (df[col2] == False)).sum()
    false_true = ((df[col1] == False) & (df[col2] == True)).sum()
    
    # Total number of rows
    total = len(df)
    
    # Calculate percentages
    conf_matrix.loc['True', 'True'] = (true_true / total) * 100
    conf_matrix.loc['False', 'False'] = (false_false / total) * 100
    conf_matrix.loc['True', 'False'] = (true_false / total) * 100
    conf_matrix.loc['False', 'True'] = (false_true / total) * 100
    
    # Ensure all values are numeric and handle any potential issues
    conf_matrix = conf_matrix.apply(pd.to_numeric, errors='coerce')  # Convert to numeric, coerce errors to NaN
    conf_matrix.fillna(0, inplace=True)  # Replace NaNs with 0 if there are any

    return conf_matrix

In [ ]:
# Define a function to create a heatmap with annotations
def create_heatmap(fig, data, row, col, showscale):
    fig.add_trace(
        go.Heatmap(
            z=data,
            text=data,  # Use the same data for annotations
            colorscale= [[0.0, '#eff3ff'], [0.05, '#9ecae1'],[0.1, '#6baed6'], [0.8, '#3182bd'], [1, '#08519c']],
            colorbar=dict(title='Scale'),
            zmin=0, zmax=100,
            showscale=showscale,
            texttemplate='%{text:.2f}%',  # Format the text annotations
            textfont=dict(size=40)
        ),
        row=row, col=col
    )
    
    fig.update_xaxes(tickvals=[0, 1], ticktext=['Detected', 'Undetected'], row=row, col=col)
    fig.update_yaxes(tickvals=[0, 1], ticktext=['Non-equivalent', 'Equivalent'], row=row, col=col)


In [ ]:
def print_confusion_matrices(df_confusion, threshold, hw, mutation_result):
    fig = make_subplots(
        rows=1, cols=len(c.metrics),
        subplot_titles=list(c.metric_names.values()),
        #x_title='Mutation analysis', y_title='Reference',
        horizontal_spacing=0.02
    )

    #fig.update_layout(annotations=[dict(font=dict(size=25))])

    print(f'Confusion matrix for {threshold} and {hw}')

    for j, metric in enumerate(c.metrics):
        df_metric = df_confusion[df_confusion['metric'] == metric]
        matrix = confusion_matrix(df_metric, 'nature', mutation_result)
        create_heatmap(fig, matrix, row=1, col=j+1, showscale=True)

    # Increase tick label font sizes
    fig.update_xaxes(tickfont=dict(size=35))
    fig.update_yaxes(tickfont=dict(size=35))

    # Hide redundant y-axis titles and tick labels
    for i in range(2, len(c.metrics) + 1):
        fig.update_yaxes(title_text='', showticklabels=False, row=1, col=i)

    fig.update_layout(
        annotations=[dict(font=dict(size=25))]
    )

    return fig


In [ ]:
folder_name = "results/RQ1/RQ1_3/"

for hw in c.hardware:
    df_hw = df[df['hardware'] == hw]
    threshold = 'I'
    df_t = df_hw[df_hw['threshold'] == threshold]
    fig = print_confusion_matrices(df_t, threshold, hw, 'ideal_label')
    c.setup_layout_and_save(fig, folder_name, f"noiseless_matrix_{threshold}_{hw}", height=550, width=3200)

    for threshold in c.thresholds:
        df_t = df_hw[df_hw['threshold'] == threshold]
        fig = print_confusion_matrices(df_t, threshold, hw, 'noisy_label')
        c.setup_layout_and_save(fig, folder_name, f"matrix_{threshold}_{hw}", height=550, width=3200)

 # RQ1.4: Accuracy, F1, Precision and recall

In [ ]:
def generate_combined_scores(df, output_filename, output_folder='results/RQ1/RQ1_4'):
    os.makedirs(output_folder, exist_ok=True)

    # Create a multi-indexed DataFrame to collect all values
    index = pd.MultiIndex.from_product(
        [[metric for metric in c.metrics.keys()], c.thresholds],
        names=["Metric", "Threshold"]
    )
    combined_df = pd.DataFrame(index=index)
    
    for metric in c.metrics:
        df_metric = df[df['metric'] == metric]
        for threshold in c.thresholds:
            df_t = df_metric[df_metric['threshold'] == threshold]
            for hw in c.hardware:
                df_hw = df_t[df_t['hardware'] == hw]

                true_labels = df_hw['nature']
                predicted_labels = df_hw['noisy_label']
                
                precision = precision_score(true_labels, predicted_labels, zero_division=0)
                recall = recall_score(true_labels, predicted_labels, zero_division=0)
                f1 = f1_score(true_labels
                              , predicted_labels, zero_division=0)
                accuracy = accuracy_score(true_labels, predicted_labels)

                # Assign scores to DataFrame
                for score_name, value in [("Accuracy", accuracy), ("F1 Score", f1), 
                                          ("Precision", precision), ("Recall", recall)]:
                    col_name = f"{score_name}-{hw}"
                    combined_df.at[(metric, threshold), col_name] = round(value, 4)

    # Reset index to make 'Metric' and 'Threshold' regular columns
    combined_df.reset_index(inplace=True)
    
    # Reorder columns: group by score before hardware
    hardware_first_cols = ['Metric', 'Threshold']
    ordered_cols = [f"{score}-{hw}" for score in c.scores for hw in c.hardware if f"{score}-{hw}" in combined_df.columns]
    combined_df = combined_df[hardware_first_cols + ordered_cols]

    # Save to file
    output_path = os.path.join(output_folder, output_filename)
    combined_df.to_csv(output_path, index=False)
    print(f"Saved all scores to {output_path}")

In [ ]:
generate_combined_scores(df, output_filename="all_scores.csv")